#Pandas Data Reshaping

#1. Pivot Table
###Dataset

In [1]:
import pandas as pd

data = {
    "Date": ["2024-01","2024-01","2024-01","2024-02","2024-02","2024-02","2024-03","2024-03"],
    "City": ["Lahore","Karachi","Lahore","Karachi","Lahore","Karachi","Lahore","Karachi"],
    "Product": ["A","A","B","A","B","B","A","B"],
    "Sales": [100,120,90,140,130,150,200,180],
    "Profit": [20,25,15,30,22,35,50,45],
    "Quantity": [10,12,9,14,13,15,20,18]
}

df = pd.DataFrame(data)
df

,Date,City,Product,Sales,Profit,Quantity
0,2024-01,Lahore,A,100,20,10
1,2024-01,Karachi,A,120,25,12
2,2024-01,Lahore,B,90,15,9
3,2024-02,Karachi,A,140,30,14
4,2024-02,Lahore,B,130,22,13
5,2024-02,Karachi,B,150,35,15
6,2024-03,Lahore,A,200,50,20
7,2024-03,Karachi,B,180,45,18


##Q1

###Create a pivot table:

####Index = City
####Columns = Product
####Values = Sales
####Aggregation = sum

In [2]:
pd.pivot_table(df, index = 'City', columns= 'Product', values='Sales', aggfunc='sum')

Product,A,B
City,,
Karachi,260,330
Lahore,300,220


##Q2 — Multiple Aggregations + Multiple Values

###Create pivot:

####Index = City
####Columns = Product
####Values = Sales & Profit
####Aggregations = sum, mean

In [3]:
pd.pivot_table(df, index = 'City', columns= 'Product', values=['Sales','Profit'], aggfunc=['sum','mean'])

sum                  mean                    
        Profit     Sales      Profit        Sales       
Product      A   B     A    B      A     B      A      B
City                                                    
Karachi     55  80   260  330   27.5  40.0  130.0  165.0
Lahore      70  37   300  220   35.0  18.5  150.0  110.0

##Q3 — Handling Duplicates + fill_value

In [4]:
pd.pivot_table(df, index='City', columns='Product',values='Sales',fill_value=0)

Product,A,B
City,,
Karachi,130.0,165.0
Lahore,150.0,110.0


##Q4 — Margins + Multi-Level Index

In [5]:
pd.pivot_table(df, index=['City','Date'], columns='Product',values='Sales', aggfunc='sum', margins=True, fill_value=0)

Product            A    B   All
City    Date                   
Karachi 2024-01  120    0   120
        2024-02  140  150   290
        2024-03    0  180   180
Lahore  2024-01  100   90   190
        2024-02    0  130   130
        2024-03  200    0   200
All              560  550  1110

##Q5 — Advanced Business Case
###“Analyze performance”

###Create pivot:

####Index = City
####Columns = Product
####Values = Sales
####Aggregation = sum
####margins=True

###Then:

####1. Convert values into percentage contribution per city
####2. Find best-performing city for each product
####3. Rank cities based on total sales

In [6]:
pivot = pd.pivot_table(
    df,
    index="City",
    columns="Product",
    values="Sales",
    aggfunc="sum",
    margins=True
)

pivot

Product,A,B,All
City,,,
Karachi,260,330,590
Lahore,300,220,520
All,560,550,1110


####1. Convert values into percentage contribution per city

In [7]:
percentage = pivot.div(pivot["All"], axis=0) * 100

percentage

Product,A,B,All
City,,,
Karachi,44.067797,55.932203,100.0
Lahore,57.692308,42.307692,100.0
All,50.450450,49.549550,100.0


###2. Find best-performing city for each product

In [8]:
pivot_no_total = pivot.drop("All")

best_city = pivot_no_total.drop("All", axis=1).idxmax()
best_city

,0
Product,
A,Lahore
B,Karachi


###3. Ranking Cities

In [9]:
ranking = pivot_no_total["All"].rank(ascending=False, method="dense")
ranking

,All
City,
Karachi,1.0
Lahore,2.0


#Topic 2: Melt
##Dataset

In [10]:

data = {
    "Date": ["2024-01","2024-02","2024-03"],
    "City": ["Lahore","Karachi","Lahore"],
    "Product_A": [100,120,130],
    "Product_B": [90,110,140],
    "Product_C": [80,100,120]
}

df = pd.DataFrame(data)
df

,Date,City,Product_A,Product_B,Product_C
0,2024-01,Lahore,100,90,80
1,2024-02,Karachi,120,110,100
2,2024-03,Lahore,130,140,120


##Q1 — Basic Melt

###Convert the dataframe into long format:

####1. Keep Date and City as identifiers
####2. Convert Product_A, Product_B, Product_C into rows
####3. Column names:
####a. Product
####b. Sales

In [17]:
pd.melt(df, id_vars= ['Date','City'], value_vars=['Product_A', 'Product_B', 'Product_C'], value_name='Sales', var_name='Product')

,Date,City,Product,Sales
0,2024-01,Lahore,Product_A,100
1,2024-02,Karachi,Product_A,120
2,2024-03,Lahore,Product_A,130
3,2024-01,Lahore,Product_B,90
4,2024-02,Karachi,Product_B,110
5,2024-03,Lahore,Product_B,140
6,2024-01,Lahore,Product_C,80
7,2024-02,Karachi,Product_C,100
8,2024-03,Lahore,Product_C,120


##Q2 — Selective Melt

##Melt only:

####Product_A and Product_B

###Keep:

####Date, City

In [20]:
pd.melt(df, id_vars= ['Date','City'], value_vars=['Product_A', 'Product_B'], value_name='Sales', var_name='Product')

,Date,City,Product,Sales
0,2024-01,Lahore,Product_A,100
1,2024-02,Karachi,Product_A,120
2,2024-03,Lahore,Product_A,130
3,2024-01,Lahore,Product_B,90
4,2024-02,Karachi,Product_B,110
5,2024-03,Lahore,Product_B,140


##Q3 — Rename & Clean Output

###After melting:

####Remove "Product_" from product names
####Convert product column into clean labels like: A, B, C

In [23]:
melting = pd.melt(df, id_vars= ['Date','City'], value_vars=['Product_A', 'Product_B'], value_name='Sales', var_name='Product')
melting['Product'] = melting['Product'].str.replace('Product_',"")
melting

,Date,City,Product,Sales
0,2024-01,Lahore,A,100
1,2024-02,Karachi,A,120
2,2024-03,Lahore,A,130
3,2024-01,Lahore,B,90
4,2024-02,Karachi,B,110
5,2024-03,Lahore,B,140


##Q4 — Melt + Aggregation

###After melting:

####1. Find total sales per Product
####2. Find average sales per City

In [25]:
melting = pd.melt(df, id_vars= ['Date','City'], value_vars=['Product_A', 'Product_B', 'Product_C'], value_name='Sales', var_name='Product')
melting

,Date,City,Product,Sales
0,2024-01,Lahore,Product_A,100
1,2024-02,Karachi,Product_A,120
2,2024-03,Lahore,Product_A,130
3,2024-01,Lahore,Product_B,90
4,2024-02,Karachi,Product_B,110
5,2024-03,Lahore,Product_B,140
6,2024-01,Lahore,Product_C,80
7,2024-02,Karachi,Product_C,100
8,2024-03,Lahore,Product_C,120


In [26]:
melting.groupby('Product')['Sales'].sum()

,Sales
Product,
Product_A,350
Product_B,340
Product_C,300


In [28]:
melting.groupby('City')['Sales'].mean()

,Sales
City,
Karachi,110.0
Lahore,110.0


##Q5 — Advanced

###“Prepare data for analysis”

###Melt the dataframe
####Create a new column:
####If Sales > 120 → "High"
####Else → "Low"
####Count how many "High" and "Low" per Product

In [29]:
melting = pd.melt(df, id_vars= ['Date','City'], value_name='Sales', var_name='Product')
melting

,Date,City,Product,Sales
0,2024-01,Lahore,Product_A,100
1,2024-02,Karachi,Product_A,120
2,2024-03,Lahore,Product_A,130
3,2024-01,Lahore,Product_B,90
4,2024-02,Karachi,Product_B,110
5,2024-03,Lahore,Product_B,140
6,2024-01,Lahore,Product_C,80
7,2024-02,Karachi,Product_C,100
8,2024-03,Lahore,Product_C,120


In [31]:
melting['category'] = melting['Sales'].apply(lambda x: 'High' if x > 120 else 'Low')
melting

,Date,City,Product,Sales,category
0,2024-01,Lahore,Product_A,100,Low
1,2024-02,Karachi,Product_A,120,Low
2,2024-03,Lahore,Product_A,130,High
3,2024-01,Lahore,Product_B,90,Low
4,2024-02,Karachi,Product_B,110,Low
5,2024-03,Lahore,Product_B,140,High
6,2024-01,Lahore,Product_C,80,Low
7,2024-02,Karachi,Product_C,100,Low
8,2024-03,Lahore,Product_C,120,Low


In [35]:
melting.groupby(['Product','category']).size()

Product    category
Product_A  High        1
           Low         2
Product_B  High        1
           Low         2
Product_C  Low         3
dtype: int64